In [ ]:
import numpy as np
import pandas as pd
import os
import duckdb

In [ ]:
data_path = os.path.expanduser("~/Documents/amazon/data")
print('STARTING EDA PROCESS')
dfs = {
    'clicks': pd.read_csv(f"{data_path}/amazon_affiliate_clicks.csv"),
    'conversions': pd.read_csv(f"{data_path}/amazon_affiliate_conversions.csv"),
    'catalog': pd.read_csv(f"{data_path}/amazon_products_catalog.csv", on_bad_lines='skip'),
    'behavior': pd.read_csv(f"{data_path}/user_behavior_analytics.csv")
}
print("🔍 1. CHECK NULL AND DUPLICATE VALUES:")
for name, df in dfs.items():
    null_count = df.isnull().sum().sum()
    dup_count = df.duplicated().sum()
    print(f"   👉 Table '{name}': {null_count} Null values | {dup_count} Duplicate rows")

print("2. CLEANING PROCESS:")
dfs['clicks']['utm_source'] = dfs['clicks']['utm_source'].fillna('organic')
dfs['clicks']['timestamp'] = pd.to_datetime(dfs['clicks']['timestamp'], errors='coerce')
dfs['conversions'] = dfs['conversions'][dfs['conversions']['order_value'] >= 0]
dfs['conversions']['timestamp'] = pd.to_datetime(dfs['conversions']['timestamp'], errors='coerce')
dfs['behavior']['time_on_page_seconds'] = dfs['behavior']['time_on_page_seconds'].fillna(0)
dfs['behavior']['scroll_depth_percentage'] = dfs['behavior']['scroll_depth_percentage'].fillna(0)
for name in dfs.keys():
    dfs[name] = dfs[name].drop_duplicates()

cleaned_path = os.path.expanduser("~/Documents/amazon/cleaned_data")

print("✅ Processed Null, Duplicates and Datetime\n")
print("💾 3. SAVE CLEANED TABLE:")
cleaned_files = {}
for name, df in dfs.items():
    save_path = f"{cleaned_path}/cleaned_{name}.csv"
    df.to_csv(save_path, index=False)
    cleaned_files[name] = save_path
    print(f"   - Saved cleaned_{name}.csv (Number of rows: {len(df):,})")

print("\n✨ DONE")

In [ ]:
import pandas as pd
import os

data_path = os.path.expanduser("~/Documents/amazon/data")

print("🛠️ EDA & CLEANING 'Amazon Sale Report.csv'...\n")


df_sales = pd.read_csv(f"{data_path}/Amazon Sale Report.csv", low_memory=False)

print("🔍 1. RAW DATA STATUS (EDA):")
print(f"   - Number of rows: {len(df_sales):,}")
print(f"   - Number of columns: {len(df_sales.columns)}")
print(f"   - Number of duplicate values: {df_sales.duplicated().sum()}")
print("\n   - Number of null values in core business sectors:")
print(df_sales[['Date', 'Amount', 'Status', 'B2B']].isnull().sum())


print("\n🧹 2. CLEANING PROCESS:")

df_sales = df_sales.drop_duplicates()

if 'Unnamed: 22' in df_sales.columns:
    df_sales = df_sales.drop(columns=['Unnamed: 22'])
    print("   - Cleaned 'Unnamed: 22'")

df_sales['Date'] = pd.to_datetime(df_sales['Date'], errors='coerce')

df_sales['Amount'] = df_sales['Amount'].fillna(0)


df_sales.columns = df_sales.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')

print("✅ DONE")

cleaned_path = os.path.expanduser("~/Documents/amazon/cleaned_data")
output_path = f"{cleaned_path}/cleaned_amazon_sale_report.csv"
df_sales.to_csv(output_path, index=False)

print('DONE')

In [ ]:
import duckdb
import pandas as pd
import os

data_path = os.path.expanduser("~/Documents/amazon/cleaned_data")
con = duckdb.connect(database=':memory:')

print("🚀 EXECUTING SQL FOR DATA MODELING...\n")

# 1. CREATE FACT_MERCHANT_SALES (B2B & PLATFORM REVENUE ANALYSIS)
query_sales = f"""
    SELECT 
        date AS event_date,
        category,
        b2b,
        status,
        COUNT(order_id) AS total_orders,
        SUM(qty) AS total_items_sold,
        SUM(amount) AS total_gmv
    FROM read_csv_auto('{data_path}/cleaned_amazon_sale_report.csv')
    WHERE date IS NOT NULL
    GROUP BY date, category, b2b, status
"""
df_fact_sales = con.execute(query_sales).df()
df_fact_sales.to_csv(f"{data_path}/Fact_Merchant_Sales.csv", index=False)
print(f"✅ Created Fact_Merchant_Sales.csv (Aggregated from 128,975 raw rows to {len(df_fact_sales):,} records)")

# 2. CREATE FACT_AFFILIATE_PERFORMANCE (CREATOR & TRAFFIC ANALYSIS)
query_affiliate = f"""
    WITH traffic AS (
        SELECT 
            click_id,
            session_id,
            CAST(timestamp AS DATE) AS event_date,
            utm_source,
            device_type
        FROM read_csv_auto('{data_path}/cleaned_clicks.csv')
    ),
    conversions AS (
        SELECT 
            click_id,
            order_value,
            commission_earned
        FROM read_csv_auto('{data_path}/cleaned_conversions.csv')
    ),
    behavior AS (
        SELECT 
            session_id,
            time_on_page_seconds
        FROM read_csv_auto('{data_path}/cleaned_behavior.csv')
    )
    
    -- Aggregate metrics by Date, Traffic Source, and Device Type
    SELECT 
        t.event_date,
        t.utm_source,
        t.device_type,
        COUNT(DISTINCT t.click_id) AS total_clicks,
        AVG(b.time_on_page_seconds) AS avg_time_on_page,
        COUNT(DISTINCT c.click_id) AS total_orders,
        SUM(COALESCE(c.order_value, 0)) AS total_affiliate_gmv,
        SUM(COALESCE(c.commission_earned, 0)) AS total_commission_cost
    FROM traffic t
    LEFT JOIN conversions c ON t.click_id = c.click_id
    LEFT JOIN behavior b ON t.session_id = b.session_id
    GROUP BY t.event_date, t.utm_source, t.device_type
"""
df_fact_affiliate = con.execute(query_affiliate).df()
df_fact_affiliate.to_csv(f"{data_path}/Fact_Affiliate_Performance.csv", index=False)
print(f"✅ Created Fact_Affiliate_Performance.csv (Aggregated {len(df_fact_affiliate)} traffic segments)")

print("\n✨ The entire Data Pipeline has been successfully executed.")

In [ ]:
import pandas as pd
import os

data_path = os.path.expanduser("~/Documents/amazon/cleaned_data")

df_affiliate = pd.read_csv(f"{data_path}/Fact_Affiliate_Performance.csv")
df_sales = pd.read_csv(f"{data_path}/Fact_Merchant_Sales.csv")

df_affiliate['event_date'] = pd.to_datetime(df_affiliate['event_date'])
df_sales['event_date'] = pd.to_datetime(df_sales['event_date'])

delta_days = df_affiliate['event_date'].min() - df_sales['event_date'].min()

df_sales['event_date'] = df_sales['event_date'] + delta_days

df_sales.to_csv(f"{data_path}/Fact_Merchant_Sales.csv", index=False)

print("✅ PERFECTLY SYNCHRONIZED!")
print(f"Affiliate start date: {df_affiliate['event_date'].min().strftime('%Y-%m-%d')}")
print(f"Sales start date: {df_sales['event_date'].min().strftime('%Y-%m-%d')}")